# Devoir Machine Learning - Stratégie de trading sur le CAC40

**Nom Prénom :**  *(à compléter)*

Prévision de la hausse du CAC40 à J+1 avec un réseau de neurones, puis backtest de la stratégie.

Installation des librairies (Colab) :

In [ ]:
%pip install -q yfinance TA-Lib tensorflow scikit-learn matplotlib

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import talib
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

## 1) Téléchargement des données CAC40

In [ ]:
df = yf.download("^FCHI", start="2017-01-01", end="2022-01-01", interval="1d", auto_adjust=False)
df.columns = df.columns.droplevel(1)

print(df.shape)
df.head()

## 2) Calcul des features

In [ ]:
df["var_j1"] = df["Close"].pct_change() * 100
df["var_j8"] = df["Close"].pct_change(8) * 100

df["adx"] = talib.ADX(df["High"], df["Low"], df["Close"], timeperiod=20)
df["rsi"] = talib.RSI(df["Close"], timeperiod=20)
stoch_k, stoch_d = talib.STOCH(df["High"], df["Low"], df["Close"], fastk_period=14, slowk_period=3)
df["stoch"] = stoch_k
df["cci"] = talib.CCI(df["High"], df["Low"], df["Close"], timeperiod=20)
df["volat"] = df["Close"].pct_change().rolling(10).std() * 100

features = ["var_j1", "var_j8", "adx", "rsi", "stoch", "cci", "volat"]
df[features].tail()

## 3) Construction du label

Label = 1 si hausse à J+1 et variation Close vs J-8 positive, 0 sinon.

In [ ]:
df["rendement"] = df["Close"].pct_change().shift(-1)
df["label"] = ((df["rendement"] > 0) & (df["var_j8"] > 0)).astype(int)

df["label"].value_counts()

## 4) Séparation train / test et normalisation

In [ ]:
df = df.dropna()

X = df[features]
y = df["label"]

n = int(len(df) * 0.8)
X_train, X_test = X[:n], X[n:]
y_train, y_test = y[:n], y[n:]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("train :", X_train.shape, "test :", X_test.shape)

## 5) et 6) Définition et compilation du réseau

In [ ]:
model = Sequential()
model.add(Dense(512, activation="elu", input_shape=(7,)))
model.add(Dropout(0.15))
model.add(Dense(256, activation="relu"))
model.add(Dropout(0.15))
model.add(Dense(128, activation="elu"))
model.add(Dropout(0.15))
model.add(Dense(32, activation="relu"))
model.add(Dropout(0.15))
model.add(Dense(1, activation="sigmoid"))

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

## 7) Entraînement

In [ ]:
model.fit(X_train, y_train, batch_size=64, epochs=25)

## 8) Prédiction sur les données de test

In [ ]:
y_proba = model.predict(X_test)
y_pred = (y_proba > 0.5).astype(int).flatten()

print("hausses prédites :", y_pred.sum(), "sur", len(y_pred))

## 9) Qualité de la prédiction

In [ ]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

## 10) Backtest de la stratégie

On achète à la clôture quand le modèle prédit une hausse, et on encaisse le rendement du lendemain. Sinon on ne fait rien.

In [ ]:
bt = df.iloc[n:].copy()
bt["pred"] = y_pred
bt["gain"] = np.where(bt["pred"] == 1, bt["rendement"], 0)

ordres = (bt["pred"] == 1).sum()
gagnants = ((bt["pred"] == 1) & (bt["rendement"] > 0)).sum()
perdants = ordres - gagnants
pnl = bt["gain"].sum() * 100
cumul = (1 + bt["gain"]).prod() - 1

print("Nombre d'ordres :", ordres)
print("Gagnants :", gagnants)
print("Perdants :", perdants)
print("Taux de reussite :", round(100 * gagnants / ordres, 1), "%")
print("P&L :", round(pnl, 2), "%")
print("Rendement cumulé :", round(cumul * 100, 2), "%")

In [ ]:
cumul = (1 + bt["gain"]).cumprod() - 1

plt.figure(figsize=(12, 6))
plt.plot(cumul * 100)
plt.title("Rendement cumulé de la stratégie")
plt.ylabel("Rendement cumulé (%)")
plt.grid(True)
plt.show()